<a href="https://colab.research.google.com/github/janithcyapa/DHCA-Framework/blob/main/Control%20Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Control Test


In [1]:

!pip uninstall -y energy-plus-utility

# %%
!pip install -q "energy-plus-utility @ git+https://github.com/janithcyapa/energy-plus-utility.git@main"
import importlib.metadata
ver = importlib.metadata.version("energy-plus-utility")
print(f"\n✅ Installed 'energy-plus-utility' version: {ver}")

# %%
from eplus import prepare_colab_eplus
prepare_colab_eplus(silent=False)

# %%
# Optional: install control (not used in this notebook but kept for compatibility)
!pip install -q control

# ## 2. Load Model (IDF + Weather)


import types, datetime, requests, io, os, gc
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from eplus.core import EPlusUtil

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

✅ Installed 'energy-plus-utility' version: 0.2.2+5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 578.3/578.3 kB 10.7 MB/s eta 0:00:00


In [2]:


OUT_DIR = "/simulation/eplus_out"
url_idf = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/5ZoneAirCooled_Exp.idf"
url_epw = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/Weather%20Files/LKA_Colombo-Katunayake.434500_SWERA.epw"

sim = EPlusUtil(verbose=3, out_dir=OUT_DIR)
sim.reset_state()
sim.delete_out_dir()
sim.clear_eplus_outputs(patterns="eplusout.*")
sim.set_model_from_url(url_idf, url_epw)

# ## 3. Register Variables for Logging

# %%
# Define all variables we want to log: real values and reference/setpoints.
# We'll use "*" for key where we want per-zone values, else specify component name.
specs = [
    # --- Zone conditions (real) ---
    {"name": "Zone Mean Air Temperature", "key": "*"},
    {"name": "Zone Mean Radiant Temperature", "key": "*"},
    {"name": "Zone Mean Air Humidity Ratio", "key": "*"},
    {"name": "Zone Air CO2 Concentration", "key": "*"},
    {"name": "Zone People Occupant Count", "key": "*"},

    # --- Supply system real values ---
    {"name": "System Node Temperature", "key": "VAV Sys 1 Outlet Node"},
    {"name": "System Node Humidity Ratio", "key": "VAV Sys 1 Outlet Node"},
    {"name": "System Node CO2 Concentration", "key": "VAV Sys 1 Outlet Node"},
    {"name": "System Node Mass Flow Rate", "key": "VAV Sys 1 Outlet Node"},

    # --- Zone air inlet (after VAV+reheat) real flows ---
    {"name": "System Node Current Density Volume Flow Rate", "key": "SPACE1-1 In Node"},
    {"name": "System Node Current Density Volume Flow Rate", "key": "SPACE2-1 In Node"},
    {"name": "System Node Current Density Volume Flow Rate", "key": "SPACE3-1 In Node"},
    {"name": "System Node Current Density Volume Flow Rate", "key": "SPACE4-1 In Node"},
    {"name": "System Node Current Density Volume Flow Rate", "key": "SPACE5-1 In Node"},

    # --- Hot water coil flows (reheat + main + OA heat) ---
    {"name": "System Node Mass Flow Rate", "key": "SPACE1-1 Zone Coil Water In Node"},
    {"name": "System Node Mass Flow Rate", "key": "SPACE2-1 Zone Coil Water In Node"},
    {"name": "System Node Mass Flow Rate", "key": "SPACE3-1 Zone Coil Water In Node"},
    {"name": "System Node Mass Flow Rate", "key": "SPACE4-1 Zone Coil Water In Node"},
    {"name": "System Node Mass Flow Rate", "key": "SPACE5-1 Zone Coil Water In Node"},
    {"name": "System Node Mass Flow Rate", "key": "Main Heating Coil 1 Water Inlet Node"},
    {"name": "System Node Mass Flow Rate", "key": "OA Heating Coil 1 Water Inlet Node"},

    # --- Chilled water coil flows ---
    {"name": "System Node Mass Flow Rate", "key": "Main Cooling Coil 1 Water Inlet Node"},
    {"name": "System Node Mass Flow Rate", "key": "OA Cooling Coil 1 Water Inlet Node"},

    # --- Plant flows ---
    {"name": "Pump Mass Flow Rate", "key": "CW CIRC PUMP"},
    {"name": "Pump Mass Flow Rate", "key": "HW CIRC PUMP"},

    # --- Chiller & Boiler status ---
    {"name": "Cooling Coil Total Cooling Rate", "key": "Main Cooling Coil 1"},
    {"name": "Boiler Heating Rate", "key": "Central Boiler"},
    {"name": "Chiller Evaporator Cooling Rate", "key": "Central Chiller"},

    # --- Outdoor environment ---
    {"name": "Site Outdoor Air Drybulb Temperature", "key": "*"},
    {"name": "Site Outdoor Air Humidity Ratio", "key": "*"},
    {"name": "Schedule Value", "key": "CO2-Outdoor-Actuated"},  # outdoor CO2 ppm

    # --- Reference/setpoint values (we will fill these from our control handler) ---
    # We'll log them manually in the logger.
]

sim.ensure_output_variables(specs, activate=True)


# ## 4. Data Logger

# %%
sim.collected_data = []
sim.current_state = {}

def state_logger(self, state):
    """Collect all real values and also our applied setpoints (stored in self.control_setpoints)."""
    if not self.exchange.api_data_fully_ready(state):
        return

    day = self.exchange.day_of_year(state)
    time_now = self.exchange.current_time(state)
    total_minutes = int(time_now * 60)
    hours, mins = divmod(total_minutes, 60)

    row = {
        "timestamp": f"Day {day:03d} {hours:02d}:{mins:02d}",
        "day": day,
        "hour": hours,
        "minute": mins,
        "time_decimal": time_now
    }

    # Helper to get variable value
    def get_val(name, key):
        handle = self.exchange.get_variable_handle(state, name, key)
        return self.exchange.get_variable_value(state, handle) if handle != -1 else np.nan

    # --- Outdoor ---
    row["T_out"] = get_val("Site Outdoor Air Drybulb Temperature", "Environment")
    row["W_out"] = get_val("Site Outdoor Air Humidity Ratio", "Environment")
    row["CO2_out"] = get_val("Schedule Value", "CO2-Outdoor-Actuated")

    # --- Supply ---
    row["T_supply"] = get_val("System Node Temperature", "VAV Sys 1 Outlet Node")
    row["W_supply"] = get_val("System Node Humidity Ratio", "VAV Sys 1 Outlet Node")
    row["CO2_supply"] = get_val("System Node CO2 Concentration", "VAV Sys 1 Outlet Node")
    row["M_supply"] = get_val("System Node Mass Flow Rate", "VAV Sys 1 Outlet Node")

    # --- Zone conditions & flows ---
    zones = ["SPACE1-1", "SPACE2-1", "SPACE3-1", "SPACE4-1", "SPACE5-1"]
    for z in zones:
        row[f"{z}_T_in"] = get_val("Zone Mean Air Temperature", z)
        row[f"{z}_T_m"]   = get_val("Zone Mean Radiant Temperature", z)
        row[f"{z}_W_in"]  = get_val("Zone Mean Air Humidity Ratio", z)
        row[f"{z}_CO2"]   = get_val("Zone Air CO2 Concentration", z)
        row[f"{z}_Occ"]   = get_val("Zone People Occupant Count", z)
        row[f"{z}_V_dot"] = get_val("System Node Current Density Volume Flow Rate", f"{z} In Node")

    # --- Reheat coil water flows ---
    for z in zones:
        row[f"{z}_reheat_water_mdot"] = get_val("System Node Mass Flow Rate", f"{z} Zone Coil Water In Node")

    # --- Main heating coil water flow ---
    row["main_heat_water_mdot"] = get_val("System Node Mass Flow Rate", "Main Heating Coil 1 Water Inlet Node")

    # --- OA heating coil water flow ---
    row["oa_heat_water_mdot"] = get_val("System Node Mass Flow Rate", "OA Heating Coil 1 Water Inlet Node")

    # --- Main cooling coil water flow ---
    row["main_cool_water_mdot"] = get_val("System Node Mass Flow Rate", "Main Cooling Coil 1 Water Inlet Node")

    # --- OA cooling coil water flow ---
    row["oa_cool_water_mdot"] = get_val("System Node Mass Flow Rate", "OA Cooling Coil 1 Water Inlet Node")

    # --- Pump flows ---
    row["cw_pump_mdot"] = get_val("Pump Mass Flow Rate", "CW CIRC PUMP")
    row["hw_pump_mdot"] = get_val("Pump Mass Flow Rate", "HW CIRC PUMP")

    # --- Coil loads ---
    row["main_cool_coil_rate"] = get_val("Cooling Coil Total Cooling Rate", "Main Cooling Coil 1")
    row["chiller_evap_rate"] = get_val("Chiller Evaporator Cooling Rate", "Central Chiller")
    row["boiler_heat_rate"] = get_val("Boiler Heating Rate", "Central Boiler")

    # --- Append our applied setpoints (stored by god_mode_control) ---
    sp = getattr(self, 'control_setpoints', {})
    row.update(sp)   # add all keys from setpoint dict

    self.collected_data.append(row)

sim.state_logger = types.MethodType(state_logger, sim)


# ## 5. Occupancy Injector (from CSV)

# %%
def preload_occupancy_csv(sim_obj, url):
    print(f"Downloading CSV from: {url}...")
    resp = requests.get(url)
    resp.raise_for_status()
    df = pd.read_csv(io.StringIO(resp.text))
    if 'timestamp' not in df.columns:
        raise ValueError("CSV must contain a 'timestamp' column.")
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp')
    midnight_start = df['timestamp'].iloc[0].normalize()
    df['rel_seconds'] = (df['timestamp'] - midnight_start).dt.total_seconds()
    sim_obj._occ_duration_sec = 86400.0
    df = df.set_index('rel_seconds')
    sim_obj._preloaded_occ_df = df.drop(columns=['timestamp'])
    zones = list(sim_obj._preloaded_occ_df.columns)
    print(f"Success! Preloaded {len(df)} rows. Loop locked to 24.00 hours.")
    print(f"Detected Source Columns: {zones}")

csv_url = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/Weather%20Files/Occupancy_Dataset.csv"
preload_occupancy_csv(sim, csv_url)

def people_injector(self, state):
    if not self.exchange.api_data_fully_ready(state) or self.exchange.warmup_flag(state):
        return
    if not hasattr(self, '_fast_injector_ready'):
        if not hasattr(self, '_preloaded_occ_df'):
            print("[Injector] ERROR: Data not preloaded.")
            self._fast_injector_ready = False
            return
        self._zone_occ_rules = {
            "SPACE1-1": {"source": "SPACE1-1", "mult": 1.0, "min": 0, "max": 5},
            "SPACE2-1": {"source": "SPACE1-1", "mult": 1.5, "min": 0, "max": 4},
            "SPACE3-1": {"source": "SPACE1-1", "mult": 0.4, "min": 0, "max": 1},
            "SPACE4-1": {"source": "SPACE1-1", "mult": 1.2, "min": 0, "max": 3},
            "SPACE5-1": {"source": "SPACE1-1", "mult": 2.0, "min": 0, "max": 6},
        }
        self._people_handles = {}
        target_zones = list(self._zone_occ_rules.keys())
        try:
            ep_people_names = self.exchange.get_object_names(state, "People") or []
        except Exception:
            ep_people_names = []
        for z in target_zones:
            matched = [p for p in ep_people_names if z.replace(" ","").lower() in p.replace(" ","").lower()]
            handles = []
            for p in matched:
                h = self.exchange.get_actuator_handle(state, "People", "Number of People", p)
                if h != -1:
                    handles.append(h)
            if handles:
                self._people_handles[z] = handles
        day = self.exchange.day_of_year(state)
        time_hr = self.exchange.current_time(state)
        self._sim_start_date = datetime.datetime(2002, 1, 1) + datetime.timedelta(days=day-1, seconds=int(time_hr*3600))
        self._fast_injector_ready = True

    if not self._fast_injector_ready or getattr(self, '_occ_duration_sec',0)==0:
        return

    day = self.exchange.day_of_year(state)
    time_hr = self.exchange.current_time(state)
    current_date = datetime.datetime(2002,1,1) + datetime.timedelta(days=day-1, seconds=int(time_hr*3600))
    elapsed_seconds = (current_date - self._sim_start_date).total_seconds()
    loop_sec = elapsed_seconds % self._occ_duration_sec
    df = self._preloaded_occ_df
    valid_indices = df.index[df.index <= loop_sec]
    target_idx = df.index[0] if len(valid_indices)==0 else valid_indices[-1]
    row = df.loc[target_idx]

    for z, handles in self._people_handles.items():
        rule = self._zone_occ_rules.get(z)
        if not rule:
            continue
        src_col = rule["source"]
        if src_col in row:
            base_val = float(row[src_col])
            if base_val == 0:
                val = 0.0
            else:
                calculated = np.ceil(base_val * rule["mult"])
                val = float(np.clip(calculated, rule["min"], rule["max"]))
            per_actuator = val / len(handles)
            for h in handles:
                self.exchange.set_actuator_value(state, h, per_actuator)

sim.people_injector = types.MethodType(people_injector, sim)


# ## 6. Outdoor CO₂ Injection

# %%
def co2_set_outdoor_ppm(self, state, value_ppm=420.0, log_every_minutes=60):
    """Keep outdoor CO₂ at a constant value using the schedule actuator"""
    if not hasattr(self, '_co2_out_handle'):
        self._co2_out_handle = self.exchange.get_actuator_handle(state,
            "Schedule:Compact", "Schedule Value", "CO2-Outdoor-Actuated")
        self._co2_log_interval = log_every_minutes
        self._co2_last_log = -999
    if self._co2_out_handle != -1:
        self.exchange.set_actuator_value(state, self._co2_out_handle, value_ppm)
    # Optional log
    if getattr(self, '_co2_last_log', -999) + self._co2_log_interval <= self.exchange.current_time(state)*60:
        self._co2_last_log = self.exchange.current_time(state)*60

sim.co2_set_outdoor_ppm = types.MethodType(co2_set_outdoor_ppm, sim)


# ## 7. God-Mode Control Handler (Set All Components)

# %%
def god_mode_control(self, state):
    """Direct override of all major HVAC components. Modify the constants below."""
    if not self.exchange.api_data_fully_ready(state) or self.exchange.warmup_flag(state):
        return

    # ============================================================
    # USER SETPOINTS – change these values to test different scenarios
    # ============================================================
    TARGET_ZONE_FLOW = 0.05          # kg/s (per zone VAV damper)
    TARGET_FAN_FLOW = 0.3            # kg/s (total supply fan)

    TARGET_MAIN_CC_WATER = 1.5       # kg/s (main cooling coil water)
    TARGET_MAIN_HC_WATER = 0.0       # kg/s (main heating coil water)
    TARGET_OA_CC_WATER = 0.2         # kg/s (OA cooling coil)
    TARGET_OA_HC_WATER = 0.0         # kg/s (OA heating coil)

    TARGET_CW_PUMP_FLOW = 3.0        # kg/s
    TARGET_HW_PUMP_FLOW = 1.0        # kg/s

    CHILLER_ON = 1.0                 # 1.0 = on, 0.0 = off
    BOILER_ON = 1.0

    # For supply temperature we keep the existing schedules (not overridden here)
    # but you could add extra actuators for that.

    # ============================================================
    # 1. VAV dampers (air mass flow)
    # ============================================================
    zones = ["SPACE1-1", "SPACE2-1", "SPACE3-1", "SPACE4-1", "SPACE5-1"]
    for z in zones:
        node_key = f"{z} ATU IN NODE"
        for c_type in ["Mass Flow Rate Setpoint",
                       "Mass Flow Rate Minimum Available Setpoint",
                       "Mass Flow Rate Maximum Available Setpoint"]:
            h = self.exchange.get_actuator_handle(state, "System Node Setpoint", c_type, node_key)
            if h != -1:
                self.exchange.set_actuator_value(state, h, TARGET_ZONE_FLOW)

    # 2. Supply fan
    h_fan = self.exchange.get_actuator_handle(state, "Fan", "Fan Air Mass Flow Rate", "SUPPLY FAN 1")
    if h_fan != -1:
        self.exchange.set_actuator_value(state, h_fan, TARGET_FAN_FLOW)

    # 3. Main cooling coil
    h_mc_on = self.exchange.get_actuator_handle(state, "Plant Component Coil:Cooling:Water",
                                                 "On/Off Supervisory", "MAIN COOLING COIL 1")
    if h_mc_on != -1:
        self.exchange.set_actuator_value(state, h_mc_on, 1.0)
    for c_type in ["Mass Flow Rate Setpoint","Mass Flow Rate Maximum Available Setpoint","Mass Flow Rate Minimum Available Setpoint"]:
        h_valve = self.exchange.get_actuator_handle(state, "System Node Setpoint", c_type,
                                                     "MAIN COOLING COIL 1 WATER INLET NODE")
        if h_valve != -1:
            self.exchange.set_actuator_value(state, h_valve, TARGET_MAIN_CC_WATER)

    # 4. Main heating coil
    h_mh_on = self.exchange.get_actuator_handle(state, "Plant Component Coil:Heating:Water",
                                                 "On/Off Supervisory", "MAIN HEATING COIL 1")
    if h_mh_on != -1:
        self.exchange.set_actuator_value(state, h_mh_on, 1.0)
    for c_type in ["Mass Flow Rate Setpoint","Mass Flow Rate Maximum Available Setpoint","Mass Flow Rate Minimum Available Setpoint"]:
        h_valve = self.exchange.get_actuator_handle(state, "System Node Setpoint", c_type,
                                                     "MAIN HEATING COIL 1 WATER INLET NODE")
        if h_valve != -1:
            self.exchange.set_actuator_value(state, h_valve, TARGET_MAIN_HC_WATER)

    # 5. OA cooling coil
    h_oc_on = self.exchange.get_actuator_handle(state, "Plant Component Coil:Cooling:Water",
                                                 "On/Off Supervisory", "OA COOLING COIL 1")
    if h_oc_on != -1:
        self.exchange.set_actuator_value(state, h_oc_on, 1.0)
    for c_type in ["Mass Flow Rate Setpoint","Mass Flow Rate Maximum Available Setpoint","Mass Flow Rate Minimum Available Setpoint"]:
        h_valve = self.exchange.get_actuator_handle(state, "System Node Setpoint", c_type,
                                                     "OA COOLING COIL 1 WATER INLET NODE")
        if h_valve != -1:
            self.exchange.set_actuator_value(state, h_valve, TARGET_OA_CC_WATER)

    # 6. OA heating coil
    h_oh_on = self.exchange.get_actuator_handle(state, "Plant Component Coil:Heating:Water",
                                                 "On/Off Supervisory", "OA HEATING COIL 1")
    if h_oh_on != -1:
        self.exchange.set_actuator_value(state, h_oh_on, 1.0)
    for c_type in ["Mass Flow Rate Setpoint","Mass Flow Rate Maximum Available Setpoint","Mass Flow Rate Minimum Available Setpoint"]:
        h_valve = self.exchange.get_actuator_handle(state, "System Node Setpoint", c_type,
                                                     "OA HEATING COIL 1 WATER INLET NODE")
        if h_valve != -1:
            self.exchange.set_actuator_value(state, h_valve, TARGET_OA_HC_WATER)

    # 7. Pumps
    h_cw_pump = self.exchange.get_actuator_handle(state, "Pump", "Pump Mass Flow Rate", "CW CIRC PUMP")
    if h_cw_pump != -1:
        self.exchange.set_actuator_value(state, h_cw_pump, TARGET_CW_PUMP_FLOW)

    h_hw_pump = self.exchange.get_actuator_handle(state, "Pump", "Pump Mass Flow Rate", "HW CIRC PUMP")
    if h_hw_pump != -1:
        self.exchange.set_actuator_value(state, h_hw_pump, TARGET_HW_PUMP_FLOW)

    # 8. Chiller / Boiler on/off
    h_chiller = self.exchange.get_actuator_handle(state, "Plant Component Chiller:Electric", "On/Off Supervisory", "CENTRAL CHILLER")
    if h_chiller != -1:
        self.exchange.set_actuator_value(state, h_chiller, CHILLER_ON)

    h_boiler = self.exchange.get_actuator_handle(state, "Plant Component Boiler:HotWater", "On/Off Supervisory", "CENTRAL BOILER")
    if h_boiler != -1:
        self.exchange.set_actuator_value(state, h_boiler, BOILER_ON)

    # Store the applied setpoints for logging
    self.control_setpoints = {
        "set_TARGET_ZONE_FLOW": TARGET_ZONE_FLOW,
        "set_TARGET_FAN_FLOW": TARGET_FAN_FLOW,
        "set_MAIN_CC_WATER": TARGET_MAIN_CC_WATER,
        "set_MAIN_HC_WATER": TARGET_MAIN_HC_WATER,
        "set_OA_CC_WATER": TARGET_OA_CC_WATER,
        "set_OA_HC_WATER": TARGET_OA_HC_WATER,
        "set_CW_PUMP_FLOW": TARGET_CW_PUMP_FLOW,
        "set_HW_PUMP_FLOW": TARGET_HW_PUMP_FLOW,
        "set_CHILLER_ON": CHILLER_ON,
        "set_BOILER_ON": BOILER_ON,
    }

sim.god_mode_control = types.MethodType(god_mode_control, sim)


# ## 8. Register All Handlers

# %%
sim.register_handlers("begin", [
    {"method_name": "state_logger"},
    {"method_name": "co2_set_outdoor_ppm", "kwargs": {"value_ppm": 420.0, "log_every_minutes": 60}},
    {"method_name": "people_injector"},
])

# Register the god-mode controller on the inside_iter hook (runs during HVAC iteration)
sim.register_handlers("inside_iter", [
    {"method_name": "god_mode_control"},
])


# ## 9. Run Simulation (Short Period for Testing)
sim.set_simulation_params(
    start=(1, 1),
    end=(1, 2),           # run for 1 day
    timestep_per_hour=4,  # 15-minute steps
    start_day_of_week="Sunday",
)

print("Starting controlled simulation...")
res = sim.run_annual()

if res == 0:
    print("Simulation complete. Converting data...")
    df = pd.DataFrame(sim.collected_data)
    # Create a proper datetime index for plotting
    sim_start = pd.Timestamp("2026-01-01 00:00:00")
    df['datetime'] = sim_start + pd.to_timedelta(df['day']-1, unit='D') + pd.to_timedelta(df['hour'] + df['minute']/60, unit='h')
    df.set_index('datetime', inplace=True)
    print("DataFrame ready.")
else:
    print("Simulation failed. Check eplusout.err")
    err_path = Path(OUT_DIR) / "eplusout.err"
    if err_path.exists():
        with open(err_path, 'r') as f:
            print(f.read()[-4000:])




# You can modify the `god_mode_control` constants, re‑run the simulation, and instantly see the impact.

Initialized StateMixin
Initialized EnergyPlus State.
Initialized IDFMixin
Initialized LoggingMixin
Initialized SimulationMixin
Initialized UtilsMixin
Initialized HandlersMixin
Initialized SQLMixin
Initialized ControlMixin
Initialized OccupancyMixin
Initialized ZoneObserverMixin
EnergyPlus state has been reset.
Output directory does not exist, nothing to delete: /simulation/eplus_out
EnergyPlus state has been reset.
Model set: IDF='/simulation/eplus_out/5ZoneAirCooled_Exp.idf', EPW='/simulation/eplus_out/LKA_Colombo-Katunayake.434500_SWERA.epw', OUT_DIR='/simulation/eplus_out'
EnergyPlus state has been reset.
Success! Preloaded 10129 rows. Loop locked to 24.00 hours.
Detected Source Columns: ['SPACE1-1']
EnergyPlus state has been reset.
Starting controlled simulation...
EnergyPlus state has been reset.
Simulation complete. Converting data...
DataFrame ready.


NameError: name 'zones' is not defined

In [6]:

def plot_comprehensive_results_split(df):
    """Create separate plots for different variable magnitude groups."""
    zones = ["SPACE1-1", "SPACE2-1", "SPACE3-1", "SPACE4-1", "SPACE5-1"]

    # ---------- Figure 1: Zone Temperatures & Humidity ----------
    fig1 = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                         subplot_titles=["Zone Temperatures (°C)", "Zone Humidity Ratios (kg/kg)",
                                         "Zone CO₂ (ppm)"])
    for z in zones:
        fig1.add_trace(go.Scatter(x=df.index, y=df[f"{z}_T_in"], name=f"{z} T_in"), row=1, col=1)
        fig1.add_trace(go.Scatter(x=df.index, y=df[f"{z}_W_in"], name=f"{z} W_in"), row=2, col=1)
        fig1.add_trace(go.Scatter(x=df.index, y=df[f"{z}_CO2"],  name=f"{z} CO₂"), row=3, col=1)
    fig1.update_yaxes(title_text="°C", row=1, col=1)
    fig1.update_yaxes(title_text="kg/kg", row=2, col=1)
    fig1.update_yaxes(title_text="ppm", row=3, col=1)
    fig1.update_layout(height=800, title="Zone Conditions")
    fig1.show()

    # ---------- Figure 2: Air / Water Flows & Occupancy ----------
    fig2 = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                         subplot_titles=["VAV Air Flows (m³/s)", "Occupancy (people)",
                                         "Reheat Coil Water Flows (kg/s)", "Main & OA Coil Water Flows (kg/s)"])
    # VAV flows (m³/s)
    for z in zones:
        fig2.add_trace(go.Scatter(x=df.index, y=df[f"{z}_V_dot"], name=f"{z} V_dot"), row=1, col=1)
    # Occupancy
    for z in zones:
        fig2.add_trace(go.Scatter(x=df.index, y=df[f"{z}_Occ"], name=f"{z} Occ"), row=2, col=1)
    # Reheat coil water flows
    for z in zones:
        fig2.add_trace(go.Scatter(x=df.index, y=df[f"{z}_reheat_water_mdot"], name=f"{z} reheat"), row=3, col=1)
    # Main / OA coil water flows
    for col in ["main_heat_water_mdot", "oa_heat_water_mdot", "main_cool_water_mdot", "oa_cool_water_mdot"]:
        fig2.add_trace(go.Scatter(x=df.index, y=df[col], name=col), row=4, col=1)
    fig2.update_yaxes(title_text="m³/s", row=1, col=1)
    fig2.update_yaxes(title_text="people", row=2, col=1)
    fig2.update_yaxes(title_text="kg/s", row=3, col=1)
    fig2.update_yaxes(title_text="kg/s", row=4, col=1)
    fig2.update_layout(height=1000, title="Flows & Occupancy")
    fig2.show()

    # ---------- Figure 3: Supply Conditions, Pump Flows & Plant Loads ----------
    fig3 = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                         subplot_titles=["Supply Conditions", "Pump Flows (kg/s)", "Coil/Plant Loads (W)"])
    # Supply conditions
    for col in ["T_supply", "W_supply", "CO2_supply", "M_supply"]:
        if col in df.columns:
            fig3.add_trace(go.Scatter(x=df.index, y=df[col], name=col), row=1, col=1)
    # Pump flows
    for col in ["cw_pump_mdot", "hw_pump_mdot"]:
        if col in df.columns:
            fig3.add_trace(go.Scatter(x=df.index, y=df[col], name=col), row=2, col=1)
    # Plant loads
    for col in ["main_cool_coil_rate", "chiller_evap_rate", "boiler_heat_rate"]:
        if col in df.columns:
            fig3.add_trace(go.Scatter(x=df.index, y=df[col], name=col), row=3, col=1)
    fig3.update_layout(height=800, title="Supply, Pumps & Plant")
    fig3.show()

plot_comprehensive_results_split(df)

In [4]:
if res == 0:
    print("Simulation complete. Converting data...")
    df = pd.DataFrame(sim.collected_data)
    sim_start = pd.Timestamp("2026-01-01 00:00:00")
    df['datetime'] = sim_start + pd.to_timedelta(df['day']-1, unit='D') + pd.to_timedelta(df['hour'] + df['minute']/60, unit='h')
    df.set_index('datetime', inplace=True)
    print("DataFrame ready.")

    # Save to CSV
    df.to_csv("simulation_results.csv", index=True)
    print("Data saved to simulation_results.csv")

Simulation complete. Converting data...
DataFrame ready.
Data saved to simulation_results.csv
